# MultiBBQ Image Synthesis: Imagen-4-Ultra

This notebook synthesizes the **MultiBBQ** benchmark images using Google's **Imagen-4-Ultra** model (`imagen-4.0-ultra-generate-001`) via **Vertex AI**.

**Reads:** the prompt/template table produced by [`../gen_template.ipynb`](../gen_template.ipynb), loaded from `../data/multibbq_template_table.csv`. It supplies the per-row generation prompts (`visual_only_ambig_prompt_w_position`, `visual_only_ambig_prompt_wo_position`, `visual_textual_prompt`) plus the contexts, questions, options and their masked variants.

**Writes:**
- Generated PNG images under `data/images/imagen4ultra_image_gen/visual_only/` and `data/images/imagen4ultra_image_gen/visual_language/`.
- A sidecar `.txt` file per image recording the prompt, contexts and Q&A used.
- Provenance CSV/JSON tables under `../data/imagen4ultra_image_gen/` (`multibbq_visual_only.*`, `multibbq_visual_language.*`) with an added `image_path` column.

> After running, move the produced `images/` folder to the project root as described in `README.md`.

**API credential:** requires Google Cloud / Vertex AI access. Authenticate with Application Default Credentials (e.g. `gcloud auth application-default login`) and set a Vertex AI **project id** and location. The project id should be supplied via an environment variable rather than hardcoded in the notebook.

## Imports

Google GenAI / Vertex AI SDKs plus Pillow for image handling.

In [ ]:
from google import genai
from google.genai import types
from PIL import Image
from io import BytesIO
import base64
import os
import json

import vertexai
from vertexai.preview.vision_models import ImageGenerationModel

## Configure Vertex AI and the Imagen-4-Ultra model

Initialize the Vertex AI client with your Google Cloud project id and location, load the `imagen-4.0-ultra-generate-001` model, and define `imagen4ultra_image_gen(prompt)` which requests a single image.

> Set your project id from an environment variable rather than hardcoding it before release.

In [ ]:
project_id = os.environ.get("GOOGLE_CLOUD_PROJECT", "<VERTEX_ID>")
location = "us-central1"
vertexai.init(project=project_id, location=location)
model = ImageGenerationModel.from_pretrained("imagen-4.0-ultra-generate-001")
def imagen4ultra_image_gen(prompt):
    response = model.generate_images(
    prompt=prompt,
    number_of_images=1,
    )
    return response

## Image save helper

`gen_and_save_images(...)` calls Imagen-4-Ultra and saves the returned image to the given path.

In [ ]:
'''
Vertext AI Imagen 4.0 Ultra API
'''
def gen_and_save_images(prompt: str, image_path: str):
    res = imagen4ultra_image_gen(prompt)
    res.images[0].save(location=image_path)
    return image_path

## Load the prompt table

Load the template/prompt table from `../gen_template.ipynb` (`../data/multibbq_template_table.csv`). The `options` columns are stringified Python lists and are parsed back with `eval`.

In [ ]:
import pandas as pd

template_df = pd.read_csv('../data/multibbq_template_table.csv')
template_df['options'] = template_df['options'].apply(eval)
template_df['options_masked'] = template_df['options_masked'].apply(eval)


## Generate the *visual-only* image set

For each template row, build the visual-only prompt (with or without positional cues), call Imagen-4-Ultra, and save the PNG to `data/images/imagen4ultra_image_gen/visual_only/` with a `.txt` sidecar capturing the prompt, contexts and Q&A. Existing images are skipped. The resulting table (with an `image_path` column) is written to `../data/imagen4ultra_image_gen/multibbq_visual_only.{csv,json}`.

> ⚠️ **Writes to the released dataset.** The next cells overwrite `../data/imagen4ultra_image_gen/`.

In [ ]:
def generate_visual(df: pd.DataFrame, ambiguity: str = 'ambig', gen_img: bool = True, position: bool = True):
    visual_df = df.copy()
    image_paths = []
    
    for idx, row in visual_df.iterrows():
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, f'../data/images/imagen4ultra_image_gen/visual_only')
        if not os.path.isdir(image_dir):
            os.makedirs(image_dir)
        
        category = row['category']
        options = row['options']
        options_masked = row['options_masked']
        
        visual_only_a = 'unknown'
        
        if ambiguity == 'ambig':
            context = 'Context: ' + row['ambig_context']
            context_masked = 'Context(masked): ' + row['ambig_context_masked']
        else:
            context = 'Context: ' + row['ambig_context'] + ' ' + row['disambig_context']
            context_masked = 'Context(masked): ' + row['ambig_context_masked'] + '\n' +  row['disambig_context_masked']
        name1 = row['name1']
        name2 = row['name2']
        
        name1_modified = name1.replace(' ', '')
        name2_modified = name2.replace(' ', '')
        image_name = f'visual_only_{category}_q{row.q_id}_c{row.c_id}_{name1_modified}_{name2_modified}'

        qna = 'Negtive Question: ' + row['neg_q'] + ' ' + 'Ans: ' + visual_only_a + '\n' + 'Nonnegtive Question: ' + row['nonneg_q'] + ' ' + 'Ans: ' + visual_only_a + '\n' + 'Options: '+ ' -'.join(row['options'])
        qna_masked = 'Negtive Question(masked): ' + row['neg_q'] + ' ' + 'Ans(masked): ' + visual_only_a + '\n' + 'Nonnegtive Question(masked): ' + row['nonneg_q'] + ' ' + 'Ans(masked): ' + visual_only_a + '\n' + 'Options(masked): ' + ' -'.join(row['options_masked'])

        if position:
            prompt = row['visual_only_ambig_prompt_w_position']
        else:
            prompt = row['visual_only_ambig_prompt_wo_position']

        image_path = os.path.join(image_dir, f'{image_name}.png')
        image_paths.append(image_path)

        if not os.path.exists(image_path):
            with open(os.path.join(image_dir, f'{image_name}.txt'), 'w') as f:
                f.write('Prompt: '+prompt+'\n\n')
                try:
                    if gen_img:
                        gen_and_save_images(prompt, image_path=image_path)
                except Exception as e:
                    print(e)
                    f.write(str(e))
                f.write(context+'\n')
                f.write(context_masked+'\n\n')
                f.write(qna+'\n\n')
                f.write(qna_masked+'\n')
    visual_df['image_path'] = image_paths        
    return visual_df


visual_data = generate_visual(template_df, 'ambig', True)
visual_data.to_csv('../data/imagen4ultra_image_gen/multibbq_visual_only.csv', index=False)
visual_data.to_json('../data/imagen4ultra_image_gen/multibbq_visual_only.json', orient='records')


## Generate the *visual-language (textual)* image set

Same flow for the visual+textual condition: build the `visual_textual_prompt` per row, generate and save each image to `data/images/imagen4ultra_image_gen/visual_language/` with its `.txt` sidecar, then write the table to `../data/imagen4ultra_image_gen/multibbq_visual_language.{csv,json}`.

In [ ]:
def generate_textual(df: pd.DataFrame, gen_img: bool = True):
    textual_df = df.copy()
    image_paths = []

    for idx, row in textual_df.iterrows():
        image_dir = os.path.join(os.curdir, f'../data/images/imagen4ultra_image_gen/visual_language')
        if not os.path.isdir(image_dir):
            os.makedirs(image_dir)
        
        category = row['category']
        options = row['options']
        options_masked = row['options_masked']
        
        neg_a = options[row['neg_label_idx']]
        nonneg_a = options[row['nonneg_label_idx']]
        neg_a_masked = options_masked[row['neg_label_idx']]
        nonneg_a_masked = options_masked[row['nonneg_label_idx']]

        name1 = row['name1']
        name2 = row['name2']
        name1_modified = name1.replace(' ', '')
        name2_modified = name2.replace(' ', '')
        image_name = f'visual_language_{category}_q{row.q_id}_c{row.c_id}_{name1_modified}_{name2_modified}'

        context = 'Ambiguous Context: ' + row['ambig_context'] + '\n' +  'Disambiguating Context: '+ row['disambig_context']
        context_masked = 'Ambiguous Context(masked): ' + row['ambig_context_masked'] + '\n' +  'Disambiguating Context(masked): ' + row['disambig_context_masked']
        qna = 'Negtive Question: ' + row['neg_q'] + ' ' + 'Ans: ' + neg_a + '\n' + 'Nonnegtive Question: ' + row['nonneg_q'] + ' ' + 'Ans:'+ nonneg_a + '\n' + 'Options: '+' -'.join(row['options'])
        qna_masked = 'Negtive Question(masked): ' + row['neg_q'] + ' ' + 'Ans(masked): ' + neg_a_masked + '\n' + 'Nonnegtive Question(masked): '+ row['nonneg_q'] + ' ' + 'Ans(masked):' + nonneg_a_masked + '\n' + 'Options(masked): '+' -'.join(row['options_masked'])

        prompt = row['visual_textual_prompt']
    
        image_path = os.path.join(image_dir, f'{image_name}.png')
        image_paths.append(image_path)
        retry = 1
        for i in range(retry):
            if i != 0:
                image_path = image_path.replace('.png', f'_{i}.png')
        if not os.path.exists(image_path):
            with open(os.path.join(image_dir, f'{image_name}.txt'), 'w') as f:
                f.write('Prompt: ' + prompt+'\n\n')
                try:
                    if gen_img:
                        gen_and_save_images(prompt, image_path=image_path)
                except Exception as e:
                    f.write(str(e))
                
                f.write(context+'\n')
                f.write(context_masked+'\n\n')
                f.write(qna+'\n\n')
                f.write(qna_masked+'\n')
    textual_df['image_path'] = image_paths
    return textual_df

textual_data = generate_textual(template_df, True)
textual_data.to_csv('../data/imagen4ultra_image_gen/multibbq_visual_language.csv', index=False)
textual_data.to_json('../data/imagen4ultra_image_gen/multibbq_visual_language.json', orient='records')
textual_data


## Done

Confirmation message pointing to the output image directories.

In [ ]:
print('Visual and Textual data generation completed.' + '\n\n' + 'You can find the generated images in the ../data/images/imagen4ultra_image_gen/visual and ../data/images/imagen4ultra_image_gen/textual directories (the repository-root images/ tree).' + '\n\n')